# 02. Model Training

End-to-end model training pipeline:
1. Stratified 80/20 train/test split (per target)
2. Bayesian hyperparameter tuning (Optuna TPE, 25 trials per learner) for LightGBM / XGBoost / CatBoost
3. Soft-voting ensemble of the tuned learners
4. Post-hoc isotonic probability calibration (5-fold CV)
5. Internal performance (AUROC, Brier) with 1,000-iteration bootstrap CIs

For end-to-end CLI usage, see `src/train_uric.py`.

Expected runtime: ~10–25 min per target on a modern laptop (8–16 cores).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from train_uric import (TARGETS, make_target, prepare,
                      build_calibrated_ensemble)
from data_utils import load_data
from eval_utils import bootstrap_metrics
from sklearn.metrics import brier_score_loss
import joblib, os

DEV_PATH = '../data/processed/development_cohort.csv'
OUT_DIR  = '../models'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
df = load_data(DEV_PATH)
print(f'Loaded {len(df):,} records')

## Train all 4 targets (≈40–90 min total depending on hardware)

In [ ]:
for name, cfg in TARGETS.items():
    print(f'\n=== {cfg["label"]} ===')
    df_t = make_target(df, name)
    X_tr, X_te, y_tr, y_te = prepare(df_t, name)
    print(f'    N train={len(X_tr):,}, test={len(X_te):,}, prevalence={y_tr.mean():.1%}')

    cal, info = build_calibrated_ensemble(X_tr, y_tr)
    y_prob = cal.predict_proba(X_te)[:, 1]
    m = bootstrap_metrics(y_te.values, y_prob, n_bootstrap=1000)
    brier = brier_score_loss(y_te, y_prob)
    print(f'    Internal AUROC: {m["AUC"]:.4f} ({m["AUC_CI"][0]:.4f}–{m["AUC_CI"][1]:.4f}), Brier: {brier:.4f}')
    joblib.dump(cal, f'{OUT_DIR}/{name}_calibrated_voting.joblib')